# 00 — Verify Connection

Confirms Databricks Connect is working against Serverless compute and that the
Unity Catalog target schemas exist (or creates them). This demo uses three schemas:
- `{UC_SCHEMA}_bronze` — raw ingested data
- `{UC_SCHEMA}_silver` — cleaned, deduplicated tables
- `{UC_SCHEMA}_gold` — star schema for analytics

In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()
print("Spark version:", spark.version)

Spark version: 4.1.0


In [2]:
spark.sql("SELECT current_user(), current_timestamp()").show(truncate=False)

+------------------------------+--------------------------+
|current_user()                |current_timestamp()       |
+------------------------------+--------------------------+
|alexander.booth@databricks.com|2026-03-19 20:54:17.993516|
+------------------------------+--------------------------+



In [3]:
UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "ticketmaster_demo")

BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"

for schema in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{schema}")
    print(f"Schema ready: {UC_CATALOG}.{schema}")

print(f"\nTarget catalog: {UC_CATALOG}")
spark.sql(f"DESCRIBE SCHEMA {UC_CATALOG}.{BRONZE_SCHEMA}").show(truncate=False)

Schema ready: alexander_booth.ticketmaster_demo_bronze
Schema ready: alexander_booth.ticketmaster_demo_silver
Schema ready: alexander_booth.ticketmaster_demo_gold

Target catalog: alexander_booth
+-------------------------+------------------------------+
|database_description_item|database_description_value    |
+-------------------------+------------------------------+
|Catalog Name             |alexander_booth               |
|Namespace Name           |ticketmaster_demo_bronze      |
|Comment                  |                              |
|Location                 |                              |
|Owner                    |alexander.booth@databricks.com|
+-------------------------+------------------------------+

